# Prueba de humo: fine-tuning de pix2tex en Colab

Este notebook solo confirma que el bucle de fine-tuning de `entrenamiento/entrenar.py` corre de punta a punta en Colab (con GPU) y guarda checkpoints. Usa `entrenamiento/dataset_validacion/` (300 formulas de entrenamiento, 40 de validacion, generadas sinteticamente) y 1 sola epoca — **no** produce un modelo con calidad real. Cuando esto funcione, pasamos al notebook de fine-tuning real con un dataset mas grande.

Antes de correr: en el menu **Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion**, elige **GPU** (T4 esta en el tier gratis).

## 1. Confirmar que hay GPU asignada

## 0. (Solo si vas a usar VS Code) Abrir un tunel SSH hacia esta VM

Corre esta celda **aqui, en el navegador de Colab**, una sola vez. Te pide una clave temporal (no la reutilices en otro lado) y al final imprime un bloque para pegar en tu `~/.ssh/config`. Con eso, VS Code (extension **Remote - SSH**) se conecta a esta misma maquina con GPU y desde ahi corres el resto de las celdas como comandos de terminal, o abres este mismo notebook en VS Code con el kernel remoto.

Si vas a seguir en el navegador de Colab en vez de VS Code, saltate esta celda.

In [ ]:
!pip install -q colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
from getpass import getpass

clave_temporal = getpass("Clave temporal para la sesion SSH (no la reutilices): ")
launch_ssh_cloudflared(password=clave_temporal)

In [ ]:
!nvidia-smi

## 2. Clonar el repo

In [ ]:
!git clone https://github.com/rimyortega55-collab/motor-OCR.git
%cd motor-OCR

## 3. Instalar dependencias

Solo lo que necesita el entrenamiento (`pix2tex` trae torch, timm, x-transformers, etc.). No instalamos el resto del proyecto (easyocr, doctr, transformers...) porque no hace falta para esta prueba.

In [ ]:
!pip install -q pix2tex wandb python-Levenshtein

## 4. Adaptar la config a las rutas de Colab

`entrenamiento/config_validacion.yaml` apunta a `.venv/Lib/site-packages/pix2tex/...` porque se escribio para el venv local en Windows. En Colab, `pip install pix2tex` deja esos mismos archivos (checkpoint pre-entrenado + tokenizer) en otra ruta, asi que la resolvemos en vivo y generamos una copia de la config con las rutas correctas.

In [ ]:
import os
import yaml
import pix2tex

base_pix2tex = os.path.dirname(pix2tex.__file__)
checkpoint = os.path.join(base_pix2tex, "model", "checkpoints", "weights.pth")
tokenizer = os.path.join(base_pix2tex, "model", "dataset", "tokenizer.json")

assert os.path.exists(checkpoint), f"No se encontro el checkpoint pre-entrenado en {checkpoint}"
assert os.path.exists(tokenizer), f"No se encontro el tokenizer en {tokenizer}"

with open("entrenamiento/config_validacion.yaml") as f:
    config = yaml.safe_load(f)

config["load_chkpt"] = checkpoint
config["tokenizer"] = tokenizer

with open("entrenamiento/config_validacion_colab.yaml", "w") as f:
    yaml.safe_dump(config, f)

print("load_chkpt:", config["load_chkpt"])
print("tokenizer: ", config["tokenizer"])

## 5. Correr la prueba de humo

`config_validacion.yaml` trae `debug: true`, pero eso solo no alcanza para desactivar wandb: `parse_args` (en `pix2tex/utils/utils.py`) sobreescribe `args.debug` con el default del propio CLI (`False`) cuando no se pasa `--debug` en la linea de comandos, asi que `args.wandb` termina en `True` igual y la version de `wandb` instalada por `pix2tex` no tiene `generate_id()` -- revienta con `AttributeError`. Por eso pasamos `--debug` explicito aca: no hace falta iniciar sesion en wandb para esta prueba.

In [ ]:
!python entrenamiento/entrenar.py --config entrenamiento/config_validacion_colab.yaml --debug

## 6. Confirmar que se guardo un checkpoint

In [ ]:
!ls -la entrenamiento/checkpoints_validacion/pix2tex_val/

Si esta celda lista al menos un archivo `.pth`, el bucle de fine-tuning corrio y guardo el modelo -- la prueba de humo paso. El siguiente paso es el notebook de fine-tuning real, con un dataset mas grande que el de validacion.